#  SystemVerilog FPGA Development with pico-dev-ice & OSS CAD Suite

---

## 1. Board Architecture: What Makes `pico-dev-ice` Unique

The **`pico-dev-ice`** pairs a Raspberry Pi Pico (RP2040) with a Lattice iCE40 UltraPlus (`iCE40UP5K-SG48`).

```
 +------------------+                   +--------------------+
 |  Raspberry Pi    |   USB DFU / SPI   |  Lattice iCE40UP5K |
 |  Pico (RP2040)   |==================>|  (CRAM only)       |
 +------------------+   Bitstream load  +--------------------+
                                                  ▲
 +------------------+                             │
 | 30.72 MHz OSC    |-----------------------------+
 | (640 x 48 kHz)   |  Hardware Clock
 +------------------+
```

### Key Engineering Details:
1. **Precision 30.72 MHz Hardware Clock:** The FPGA does *not* get its clock from the RP2040 or an unstable internal RC oscillator. It is clocked by a dedicated **30.72 MHz crystal oscillator** on the board. This frequency is exactly $640 \times 48\text{ kHz}$, making integer decimation down to baseband audio seamless.
2. **Volatile CRAM Only (No FPGA Flash):** There is no SPI flash memory attached directly to the FPGA. Bitstreams are streamed into FPGA SRAM (**CRAM**). When power is removed, the FPGA unloads.
3. **Dual Programming Options:**
   * **DFU via USB (`dfu-util`):** Fast command-line workflow communicating through the Pico's DFU interface directly into the FPGA CRAM.
   * **MicroPython:** Dynamically load bitstreams from the Pico's filesystem using the `pico-ice-micropython` firmware.

---

## 2. Setting Up OSS CAD Suite

Download the latest release for your platform from the [YosysHQ OSS CAD Suite Releases](https://github.com/YosysHQ/oss-cad-suite-build/releases/latest).

---

### 🐧 Linux Setup

1. **Extract the archive:**
   ```bash
   tar -xvzf ./oss-cad-suite-linux-x64-*.tgz -C ~/
   ```
2. **Add `udev` permissions for the Pico DFU interface:**
   ```bash
   echo 'ATTRS{idVendor}=="1209", ATTRS{idProduct}=="b1c0", MODE="0666", GROUP="plugdev"' | sudo tee /etc/udev/rules.d/99-pico-ice.rules
   sudo udevadm control --reload-rules
   sudo udevadm trigger
   ```
3. **Activate the environment:**
   ```bash
   source ~/oss-cad-suite/environment
   ```

---

### 🍏 macOS Setup

1. **Extract the archive:**
   ```bash
   tar -xvzf ./oss-cad-suite-darwin-*.tgz -C ~/
   ```
2. **Remove quarantine restrictions:**
   ```bash
   xattr -r -d com.apple.quarantine ~/oss-cad-suite
   ```
3. **Activate the environment:**
   ```bash
   source ~/oss-cad-suite/environment
   ```

---

### 🪟 Windows Setup

1. Extract the downloaded archive (e.g., to `C:\oss-cad-suite`).
2. Run **`C:\oss-cad-suite\start.bat`** to open a command prompt with all paths set.
3. **USB Driver:** If `dfu-util -l` does not see your board, run [Zadig](https://zadig.akeo.ie/), select **Options → List All Devices**, click on the **iCE40 DFU** interface, and install the **WinUSB** driver.

---

## 3. Verify Your Environment

Run this cell to verify your tools are active:

In [21]:
# Cell Type: Code
import os
import sys

# Add OSS CAD Suite to Jupyter's environment PATH
if sys.platform == "win32":
    oss_bin = r"C:\oss-cad-suite\bin"
else:
    oss_bin = os.path.expanduser("~/oss-cad-suite/bin")

if oss_bin not in os.environ["PATH"]:
    os.environ["PATH"] = oss_bin + os.pathsep + os.environ["PATH"]

# Test that Jupyter can now find them:
!which yosys || where yosys
!yosys --version
!nextpnr-ice40 --version
!dfu-util --version

/home/frohro/oss-cad-suite/bin/yosys
Yosys 0.69+173 (git sha1 53f1cdd34-dirty, Release, Clang /usr/bin/clang++ 21.1.8)
"nextpnr-ice40" -- Next Generation Place and Route (Version nextpnr-0.11.1-40-geb4f15c3)
dfu-util 0.11-dev

Copyright 2005-2009 Weston Schmidt, Harald Welte and OpenMoko Inc.
Copyright 2010-2021 Tormod Volden and Stefan Schmidt
This program is Free Software and has ABSOLUTELY NO WARRANTY
Please report bugs to https://sourceforge.net/p/dfu-util/tickets/



---

## 4. The Programs You Will Use

```
SystemVerilog (.sv) ──► verilator (Linting & Sim)
        │
        ▼
      yosys          (Synthesis to Netlist .json)
        │
        ▼
   nextpnr-ice40     (Place & Route for up5k-sg48 + .pcf)
        │
        ▼
      icepack        (Converts routing to .bin)
        │
        ▼
      Pico SPI using MicroPython ice.fpga()       (Streams .bin to FPGA CRAM)
```

1. **`verilator` (Simulation & Linting):** Fast compile-time verification. It catches unassigned wires, out-of-range bit slices, and race conditions before you attempt synthesis.
2. **`yosys` (RTL Synthesis):** Translates your SystemVerilog logic into iCE40 primitive gates (LUTs, Flip-Flops, and DSP blocks).
3. **`nextpnr-ice40` (Place & Route):** Maps the logic cells to physical silicon coordinates and routes wires between them while meeting the 30.72 MHz timing constraint.
4. **`icepack` (Bitstream Packer):** Converts the ASCII placement map into the binary `.bin` bitstream.
5. **` MicroPython ice.fpga()` (FPGA Flasher):** Sends the `.bin` over USB to the Pico's DFU runtime to configure the FPGA.

---

## 5. SystemVerilog Blinky Tutorial

Let's build a clock divider and RGB status indicator driven by the **30.72 MHz** hardware oscillator.

### Step 5.1: Create Pin Constraints (`pico_dev_ice.pcf`)

Map your hardware clock and the RGB LED pins:

In [27]:
%%writefile pico_dev_ice.pcf
# 30.72 MHz On-Board Oscillator (replace with your board's clock pin)
set_io clk        37

# Dedicated iCE40UP5K RGB Driver Pins (SG48 Package)
set_io led_red   39
set_io led_yellow  40
set_io led_green    41

Overwriting pico_dev_ice.pcf


---

### Step 5.2: Create the SystemVerilog Source (`top.sv`)

At 30.72 MHz:
* $30,720,000 \text{ cycles/sec} \div 2^{24} \approx 1.83\text{ Hz}$ (Bit 23 toggles roughly every 0.27s).
* Bit 24 toggles every ~0.55s.

In [28]:
%%writefile top.sv
`default_nettype none

// 1. Simulation/Linting stub for Verilator
// Yosys ignores this completely, but Verilator uses it to understand port directions.
`ifdef VERILATOR
/* verilator lint_off DECLFILENAME */
/* verilator lint_off UNUSEDPARAM */
/* verilator lint_off UNUSEDSIGNAL */
module SB_RGBA_DRV #(
    parameter CURRENT_MODE = "0b0",
    parameter RGB0_CURRENT = "0b000000",
    parameter RGB1_CURRENT = "0b000000",
    parameter RGB2_CURRENT = "0b000000"
)(
    input  logic CURREN,
    input  logic RGBLEDEN,
    input  logic RGB0PWM,
    input  logic RGB1PWM,
    input  logic RGB2PWM,
    output logic RGB0,
    output logic RGB1,
    output logic RGB2
);
    assign RGB0 = RGB0PWM;
    assign RGB1 = RGB1PWM;
    assign RGB2 = RGB2PWM;
endmodule
/* verilator lint_on DECLFILENAME */
/* verilator lint_on UNUSEDPARAM */
/* verilator lint_on UNUSEDSIGNAL */
`endif

module top (
    input  logic clk,          // 30.72 MHz Hardware Oscillator (Pin 37)
    output logic led_red,      // Pin 39 (RGB0)
    output logic led_yellow,   // Pin 40 (RGB1)
    output logic led_green     // Pin 41 (RGB2)
);

    // Tell Verilator that this inline initialization is intentional for FPGA power-on
    /* verilator lint_off PROCASSINIT */
    logic [24:0] counter = 25'd0;
    /* verilator lint_on PROCASSINIT */

    always_ff @(posedge clk) begin
        counter <= counter + 1'b1;
    end

    // Dedicated iCE40 Hard IP RGB Current-Sink Driver
    SB_RGBA_DRV #(
        .CURRENT_MODE("0b1"),       // Half-current mode
        .RGB0_CURRENT("0b000001"),  // ~4 mA current
        .RGB1_CURRENT("0b000001"),
        .RGB2_CURRENT("0b000001")
    ) rgb_driver_inst (
        .CURREN(1'b1),              // Master driver enable
        .RGBLEDEN(1'b1),            // LED enable
        .RGB0PWM(~counter[24]),     // Red    (Pin 39 / RGB0)
        .RGB1PWM(counter[24]),      // Yellow (Pin 40 / RGB1)
        .RGB2PWM(counter[23]),      // Green  (Pin 41 / RGB2 - toggles twice as fast)
        .RGB0(led_red),
        .RGB1(led_yellow),
        .RGB2(led_green)
    );

endmodule

Overwriting top.sv


#### SystemVerilog Details:
Here is a complete, line-by-line guide explaining the syntax, the hardware it represents, and why each construct is used.

---

##### Part 1: Compiler Directives & The Simulation Stub

```systemverilog
`default_nettype none
```
* **Line 1:** 
  * The backtick (`` ` ``) denotes a **compiler directive** (preprocessor instruction).
  * In classic Verilog, if you misspelled a signal name, the compiler silently invented a new 1-bit `wire` with no driver. 
  * Setting this to `none` forces the compiler to flag an error if you ever use a signal that hasn't been explicitly declared (`logic`, `wire`, etc.). It catches 90% of beginner typos.

```systemverilog
`ifdef VERILATOR
```
* **Line 5:** 
  * Conditional compilation. "If the macro `VERILATOR` is defined, include the following code; otherwise, ignore it completely."
  * Verilator automatically defines this when running; Yosys does not.

```systemverilog
/* verilator lint_off DECLFILENAME */
/* verilator lint_off UNUSEDPARAM */
/* verilator lint_off UNUSEDSIGNAL */
```
* **Lines 6–8:** 
  * Verilator **lint pragmas** (compiler directives in comments).
  * `DECLFILENAME`: Suppresses the warning that the filename (`top.sv`) doesn't match this module name (`SB_RGBA_DRV`).
  * `UNUSEDPARAM` & `UNUSEDSIGNAL`: Suppress warnings about parameters and input pins that this stub accepts but doesn't actually use in logic.

```systemverilog
module SB_RGBA_DRV #(
    parameter CURRENT_MODE = "0b0",
    parameter RGB0_CURRENT = "0b000000",
    parameter RGB1_CURRENT = "0b000000",
    parameter RGB2_CURRENT = "0b000000"
)(
```
* **Lines 9–14:**
  * Defines a **parameterized module** using ANSI C-style port lists.
  * The `#(...)` block declares module **parameters** (compile-time constants). These mirror the parameters the real silicon primitive expects.

```systemverilog
    input  logic CURREN,
    input  logic RGBLEDEN,
    input  logic RGB0PWM,
    input  logic RGB1PWM,
    input  logic RGB2PWM,
    output logic RGB0,
    output logic RGB1,
    output logic RGB2
);
```
* **Lines 15–23:**
  * The **port list** for the module.
  * **`input logic` / `output logic`:** SystemVerilog's universal data type. You no longer have to worry about the confusing classic Verilog rules of `wire` vs. `reg`.

```systemverilog
    assign RGB0 = RGB0PWM;
    assign RGB1 = RGB1PWM;
    assign RGB2 = RGB2PWM;
endmodule
```
* **Lines 24–27:**
  * **`assign`:** Continuous combinational assignment. 
  * In this mock simulation model, whatever logic level enters `RGBxPWM` immediately passes straight through to `RGBx`.
  * **`endmodule`:** Closes the `SB_RGBA_DRV` module definition.

```systemverilog
/* verilator lint_on DECLFILENAME */
/* verilator lint_on UNUSEDPARAM */
/* verilator lint_on UNUSEDSIGNAL */
`endif
```
* **Lines 28–31:**
  * Re-enables the lint checks for the rest of your actual design, and closes the `` `ifdef `` block.

---

#### Part 2: The Top-Level Module

```systemverilog
module top (
    input  logic clk,          // 30.72 MHz Hardware Oscillator (Pin 37)
    output logic led_red,      // Pin 39 (RGB0)
    output logic led_yellow,   // Pin 40 (RGB1)
    output logic led_green     // Pin 41 (RGB2)
);
```
* **Lines 33–38:**
  * Declares your top-level hardware entity named `top`.
  * **`input logic clk`:** A 1-bit input driven by the physical 30.72 MHz crystal on your board.
  * **`output logic ...`:** Three 1-bit output signals that will be routed to the physical LED pins via your `.pcf` file.

---

#### Part 3: The Clock Divider / Counter

```systemverilog
    /* verilator lint_off PROCASSINIT */
    logic [24:0] counter = 25'd0;
    /* verilator lint_on PROCASSINIT */
```
* **Lines 40–42:**
  * **`logic [24:0] counter`:** Declares a 25-bit wide bus/vector. Bit 24 is the Most Significant Bit (MSB), bit 0 is the Least Significant Bit (LSB). Total width = 25 bits.
  * **`= 25'd0;`:** Number literal syntax: `[size]'[base][value]`. This means a 25-bit wide, decimal value of 0. In an FPGA, this initializes the physical flip-flops to `0` upon power-up configuration.
  * The `lint_off PROCASSINIT` suppresses the warning Verilator emits when you initialize a variable inline that is also driven inside an `always` block.

```systemverilog
    always_ff @(posedge clk) begin
        counter <= counter + 1'b1;
    end
```
* **Lines 44–46:**
  * **`always_ff`:** A dedicated SystemVerilog keyword that guarantees to the synthesis tool: *"This block must synthesize strictly into clocked Flip-Flops (sequential logic)."* (In old Verilog, people used generic `always @(...)` which could accidentally create latches).
  * **`@(posedge clk)`:** Triggered only on the rising edge (transition from 0 to 1) of the 30.72 MHz clock.
  * **`<=` (Non-blocking assignment):** **Always used for sequential flip-flop logic.** It schedules the register update to occur after the clock edge, preventing race conditions between flip-flops.
  * **`1'b1`:** A 1-bit binary literal with a value of `1`.

---

#### Part 4: Instantiating the Silicon Primitive (`SB_RGBA_DRV`)

In SystemVerilog, instantiating a module uses the syntax:  
`ModuleName #( .PARAM(val) ) InstanceName ( .PORT(signal) );`

```systemverilog
    SB_RGBA_DRV #(
        .CURRENT_MODE("0b1"),       // Half-current mode
        .RGB0_CURRENT("0b000001"),  // ~4 mA current
        .RGB1_CURRENT("0b000001"),
        .RGB2_CURRENT("0b000001")
    ) rgb_driver_inst (
```
* **Lines 48–54:**
  * **`SB_RGBA_DRV #(`:** Tells Yosys we are overriding default parameters for this instance.
  * **`.CURRENT_MODE("0b1")`:** Sets the internal reference to half-current mode.
  * **`.RGBx_CURRENT(...)`:** Sets the analog current sink to ~4 mA per LED channel.
  * **`) rgb_driver_inst (`:** Gives this specific hardware instance a unique name (`rgb_driver_inst`) and begins the port connection list.

```systemverilog
        .CURREN(1'b1),              // Master driver enable
        .RGBLEDEN(1'b1),            // LED enable
```
* **Lines 55–56:**
  * **`.PORT_NAME(SIGNAL)`:** Named port association. Connects the module's internal port (`.CURREN`) to your local signal or constant (`1'b1`).
  * Connecting `1'b1` ties these control lines permanently to logic HIGH ($V_{DD}$), keeping the driver always powered on and enabled.

```systemverilog
        .RGB0PWM(~counter[24]),     // Red    (Pin 39 / RGB0)
        .RGB1PWM(counter[24]),      // Yellow (Pin 40 / RGB1)
        .RGB2PWM(counter[23]),      // Green  (Pin 41 / RGB2 - toggles twice as fast)
```
* **Lines 57–59:**
  * Feeds our counter bits into the LED controls:
  * **`counter[23]`:** Bit-slicing. Taps bit 23 of the counter, which toggles every ~0.27 seconds.
  * **`counter[24]`:** Taps bit 24, which toggles every ~0.55 seconds.
  * **`~counter[24]`:** The bitwise **NOT** operator. When bit 24 is `0`, Red is `1` (ON). When bit 24 is `1`, Red is `0` (OFF). This creates alternating phases between the Red and Yellow LEDs.

```systemverilog
        .RGB0(led_red),
        .RGB1(led_yellow),
        .RGB2(led_green)
    );
endmodule
```
* **Lines 60–64:**
  * Connects the driver's dedicated analog output lines directly to our module's top-level pins (`led_red`, `led_yellow`, `led_green`), which connect to pins 39, 40, and 41 on the chip.
  * **`);`** closes the module instantiation.
  * **`endmodule`** closes the `top` module.



---

### Step 5.3: Lint with Verilator

Catch syntax errors and dangling signals:

In [29]:
!verilator --lint-only top.sv

- V e r i l a t i o n   R e p o r t: Verilator 5.053 devel rev v5.052-273-g71c6638cc (mod)
- Verilator: Built from 0.034 MB sources in 4 modules, into 0.016 MB in 3 C++ files needing 0.000 MB
- Verilator: Walltime 0.005 s (elab=0.000, cvt=0.002, bld=0.000); cpu 0.005 s on 1 threads; allocated 19.555 MB


---

### Step 5.4: Synthesize with Yosys

Compile SystemVerilog into an iCE40 netlist:

In [30]:
# Cell Type: Code
!yosys -p "read_verilog -sv top.sv; synth_ice40 -top top -json top.json"


 /----------------------------------------------------------------------------\
 |  yosys -- Yosys Open SYnthesis Suite                                       |
 |  Copyright (C) 2012 - 2026  Claire Xenia Wolf <claire@yosyshq.com>         |
 |  Distributed under an ISC-like license, type "license" to see terms        |
 \----------------------------------------------------------------------------/
 Yosys 0.69+173 (git sha1 53f1cdd34-dirty, Release, Clang /usr/bin/clang++ 21.1.8)

-- Running command `read_verilog -sv top.sv; synth_ice40 -top top -json top.json' --

1. Executing Verilog-2005 frontend: top.sv
Parsing SystemVerilog input from `top.sv' to AST representation.
Generating RTLIL representation for module `\top'.
Successfully finished Verilog frontend.

2. Executing SYNTH_ICE40 pass.

2.1. Executing Verilog-2005 frontend: /home/frohro/oss-cad-suite/lib/../share/yosys/ice40/cells_sim.v
Parsing Verilog input from `/home/frohro/oss-cad-suite/lib/../share/yosys/ice40/cells_sim.v' to

---

### Step 5.5: Place and Route with `nextpnr-ice40`

Fit the design to the `iCE40UP5K` in the `sg48` package:

In [31]:
# Cell Type: Code
!nextpnr-ice40 --up5k --package sg48 --pcf pico_dev_ice.pcf --json top.json --asc top.asc

Info: constrained 'clk' to bel 'X13/Y31/io0'
Info: constrained 'led_red' to bel 'X4/Y31/io0'
Info: constrained 'led_yellow' to bel 'X5/Y31/io0'
Info: constrained 'led_green' to bel 'X6/Y31/io0'

Info: Packing constants..
Info: Packing IOs..
Info: led_red use by SB_RGBA_DRV/SB_RGB_DRV rgb_driver_inst, not creating SB_IO
Info: led_green use by SB_RGBA_DRV/SB_RGB_DRV rgb_driver_inst, not creating SB_IO
Info: led_yellow use by SB_RGBA_DRV/SB_RGB_DRV rgb_driver_inst, not creating SB_IO
Info: Packing LUT-FFs..
Info:        1 LCs used as LUT4 only
Info:       25 LCs used as LUT4 and DFF
Info: Packing non-LUT FFs..
Info:        0 LCs used as DFF only
Info: Packing carries..
Info:        0 LCs used as CARRY only
Info: Packing indirect carry+LUT pairs...
Info:        0 LUTs merged into carry LCs
Info: Packing RAMs..
Info: Placing PLLs..
Info: Packing special functions..
Info:   constrained SB_RGBA_DRV 'rgb_driver_inst' to X0/Y30/rgba_drv_0
Info: Packing PLLs..
Info: Promoting globals..
Info: pro

---

### Step 5.6: Generate Bitstream

Pack the ASCII route output into a binary file:

In [32]:
# Cell Type: Code
!icepack top.asc top.bin
print("Bitstream top.bin generated successfully!")

Bitstream top.bin generated successfully!


---

## 6. Loading the Bitstream onto `pico-dev-ice`

Because the board has no FPGA flash, **all bitstreams are loaded directly into CRAM**.


### Flashing via MicroPython

If your Pico is running your `pico-ice-micropython` runtime:

1. Copy `top.bin` onto the Pico's flash filesystem (using Thonny, `mpremote`, or file copy).
2. From the MicroPython REPL or a script:

### Using mpremote

### Method 1: The Direct Terminal / Jupyter Approach (Simplest)

You can copy the binary and execute your exact Python sequence in two quick commands:

#### From the Terminal:
```bash
# 1. Copy the bitstream from your computer into the Pico's root filesystem
mpremote cp top.bin :bitstream.bin

# 2. Tell the Pico to initialize the pins and load the CRAM
mpremote exec "from machine import Pin; import ice; fpga = ice.fpga(cdone=Pin(21), creset=Pin(22), cram_cs=Pin(5), cram_mosi=Pin(7), cram_sck=Pin(6)); fpga.start(); fpga.cram(open('bitstream.bin', 'br'))"
```

#### Inside a Jupyter Notebook Cell:
```python
# Cell Type: Code
!mpremote cp top.bin :bitstream.bin
!mpremote exec "from machine import Pin; import ice; fpga = ice.fpga(cdone=Pin(21), creset=Pin(22), cram_cs=Pin(5), cram_mosi=Pin(7), cram_sck=Pin(6)); fpga.start(); fpga.cram(open('bitstream.bin', 'br')); print('FPGA Loaded!')"
```

---

### Method 2: Using a Local Script (`mpremote run`)

If you don't want a long one-liner command, you can keep a clean helper script on your PC named **`load_fpga.py`**:

```python
# load_fpga.py (saved on your PC)
from machine import Pin
import ice

print("Configuring FPGA...")
fpga = ice.fpga(
    cdone=Pin(21),
    creset=Pin(22),
    cram_cs=Pin(5),
    cram_mosi=Pin(7),
    cram_sck=Pin(6)
)

fpga.start()
with open("bitstream.bin", "br") as f:
    fpga.cram(f)

print("FPGA loaded and running!")
```

Then, whenever you or your students compile a new bitstream, you just run:

```bash
mpremote cp top.bin :bitstream.bin
mpremote run load_fpga.py
```
*(The `run` command streams `load_fpga.py` over USB into the Pico's RAM and executes it immediately).*

---

### One Quick Tip
Make sure **Thonny is closed** when using `mpremote`. If Thonny is open, it holds an exclusive lock on the USB serial port, causing `mpremote` to report a `Device or resource busy` or `could not open port` error.

---

## 7. Quick Reference Cheat Sheet

| Step | Command |
| :--- | :--- |
| **Lint SV** | `verilator --lint-only -Wall top.sv` |
| **Synthesize** | `yosys -p "read_verilog -sv top.sv; synth_ice40 -top top -json top.json"` |
| **Place & Route** | `nextpnr-ice40 --up5k --package sg48 --pcf pico_dev_ice.pcf --json top.json --asc top.asc` |
| **Pack Bitstream** | `icepack top.asc top
| **Load (MicroPython)** | `pico_ice.fpga.load(open("top.bin", "rb"))` |